# 美国国会图书馆（Library of Congress）联机目录 SRU 实验

**课件位置**：本笔记本位于课程文件夹 **`上机实验/01周/`**；抓取结果写入同级 **`data/lc_sru_export/`**。

本练习带你用「已经写好的步骤」从美国国会图书馆的**联机目录**里取出书目数据，并观察 **MARC** 里常见字段的含义。

- **即使你几乎不会编程**，只要会按顺序运行笔记本里的格子，也能得到打印结果和下载下来的文件，并完成课程要求的观察与讨论。
- **若你已经会 Python**，可以把本笔记本当成可直接扩展的脚本骨架。

下面请先阅读 **「开始之前：写给还不太熟悉编程的同学」**，再从上往下运行各单元格。

---

**美国国会图书馆**（Library of Congress，缩写 **LC**）是美国国会下属的国立图书馆。它对公众开放了**多种**机器访问方式（官网集中说明在 [APIs at the Library of Congress](https://www.loc.gov/apis/)）；**本笔记本只用到其中一种**：通过 **SRU** 请求 **联机目录 LCDB**，得到 **XML** 里的 **MARC 21** 书目记录。

**你需要自备**：稳定的互联网；可运行 **Jupyter Notebook**（或 JupyterLab）的环境（许多学校机房或 Anaconda 已预装）。运行期间脚本会自动停顿几秒再发下一次请求，以减少对美国国会图书馆服务器的压力；请遵守课程与官网的合理使用要求。


## 开始之前：写给还不太熟悉编程的同学

### 这个文件（`.ipynb`）是什么？

你可以把它想成一篇**可以互动的讲义**：既有文字说明，又有一些「代码格子」。代码已经写好，**你的任务主要是运行它、读输出、对照讲义思考问题**，而不是从零写程序。

### 怎么「运行」？

1. 用 Jupyter Notebook / JupyterLab / VS Code（带 Jupyter 插件）打开本文件。  
2. 选中某个**代码**格子（一般是浅灰背景），按 **`Shift + Enter`**（或点工具栏里的「运行」）：会先执行这一格，再自动跳到下一格。  
3. **请从上到下按顺序运行**。第一次使用时，务必先运行「**导入库与工具函数**」那一格（里面有 `import …` 和 `def sru_search…`），再运行后面的对比实验和下载——否则后面的格子会提示「找不到某某名称」。

### 我需要读懂每一行代码吗？

**不需要。** 代码旁边的注释和讲义里的表格，是为了日后你想深挖时可以对照。最低限度上，你只要能够：

- 辨认：**哪里是打印出来的统计结果**（命中多少条等）；  
- 找到：**下载下来的文件夹和 CSV 文件**在什么位置（一般在 **`data/lc_sru_export/`**（与笔记本并列的 **data** 文件夹））。

### 如果跑出红色报错怎么办？

常见情况（课堂上可以逐项排查）：

| 现象 | 可能原因 |
|------|----------|
| `NameError: name 'sru_search' is not defined` | 还没运行「工具函数」那一格，或顺序乱了 | 
| 超时、`URLError`、`Connection timed out` | 当前网络访问不了美国国会图书馆服务器，换网络/稍后重试 |
| 长时间停在某一格 | 正在下载较多数据，属正常；若超过几分钟仍无输出，可询问教师 |
| `First record position out of range`（§5 分页） | SRU 返回了「起点越界」类诊断。请使用**已更新的** §5 代码：会停止翻页并保留已下载页；可稍后重试或缩小/拆分检索式 |

### 作业可以交什么？（示例）

教师可根据课程调整；对不写代码的同学，通常可以交：

- **截图**：检索对比、分页下载完成后的控制台输出（能看出命中数与保存路径）；  
- **文件**：`data/lc_sru_export/xml_pages/` 里某一页 `.xml`，或 `书目摘要.csv` / `书目摘要.parquet` / **`marc_flat.parquet`**（MARC 全字段长表，若不让交原件，可只交摘录）；  
- **文字**：用自己的话说明「关键词检索」和「按题名/主题字段检索」命中数为何不同、你在 MARC 里注意到了哪些出版或主题信息。

### 想在本机安装 Jupyter（可选）

- 安装 **Anaconda**（自带 Jupyter）或 **Miniconda**，再在终端执行：`jupyter notebook`；  
- 或在你熟悉的编辑器里安装 **Jupyter 插件** 打开本文件。具体步骤以学校实验指导为准。

**文件会保存在哪里**：运行 §5 后，在 **`上机实验/01周/data/lc_sru_export/`** 下会有 `xml_pages/`（分页原始 XML）、`书目摘要.csv`、`书目摘要.parquet`、**`marc_flat.parquet`**（每条书目多行，含 650 等全部子字段）。


## 1. 美国国会图书馆有哪些 API？本实验用的是哪一种？

**美国国会图书馆**（Library of Congress，常缩写 **LC**）在门户 **[APIs at the Library of Congress](https://www.loc.gov/apis/)** 上集中说明「如何用程序访问」它的数据与服务。官网上**不会给出一个永远不变的「API 总数」**——新服务会增、旧接口会调整——因此要学会 **按层次理解**，需要时再点开子文档。

若用「能交互的方式」来归类（便于课堂记忆），通常可以记成下面几块（**数量级是结构示意，以官网为准**）：

| 层次 | 大约几种 / 怎么理解 | 典型用途（举例） |
|------|---------------------|------------------|
| **loc.gov JSON/YAML API** | **一套**协议下有多组端点（`/search/`、`/collections/`、`/item/…` 等） | 检索 **loc.gov 网站与数字馆藏** 的呈现内容，拿 JSON/YAML（见 [JSON/YAML for LoC.gov](https://www.loc.gov/apis/json-and-yaml/)） |
| **Microservices（微服务）** | 文档里常归纳为 **三大类**：文本、图像、流媒体 | 例如全文片段、**IIIF** 图像、音视频播放相关能力 |
| **[Additional APIs and Services](https://www.loc.gov/apis/additional-apis/)** | **一页清单**，下列 **多项彼此独立** 的接口（条目数会增减；课堂上可说「大约十种量级」） | 例如 **Congress.gov**（国会立法）、**Chronicling America**（老报纸）、**Linked Data**（规范档/关联数据）、**PPOC**（印刷品与照片）、**NLS BARD**（视障服务）、以及 **SRU / Z39.50（联机目录与部分检索）** 等 |

**本笔记本只占其中一条路径**：通过 **SRU** 访问 **联机目录 LCDB**，返回 **XML**，书目明细是 **MARC 21**。它与 **loc.gov JSON API**（面向网站与数字馆藏门户）**不是同一套东西**——后者更像「网站与数字化藏品」的机器接口；这里是「**图书馆目录记录**」的机器检索。

若要和学生一起做「清点」练习，可布置：**打开 Additional APIs 页面，统计当前列表里有多少个带独立文档的链接**——这就是「当下有多少种并列的补充 API」的一种可操作答案。

---

## 2. SRU 与本实验用的端点

**SRU**（Search/Retrieval via URL）是一种基于 HTTP 的检索协议：把检索式写在 URL 参数里，服务器返回 **XML**，里面的书目身体是 **MARC XML**。技术说明见 [Search/Retrieval via URL](https://www.loc.gov/apis/additional-apis/search-retrieval-via-url/)，服务器列表见 [LC SRU Servers](https://www.loc.gov/standards/sru/resources/lcServers.html)。

本实验使用的 **美国国会图书馆联机目录（LCDB）** 基址为（HTTP，端口 210）：

`http://lx2.loc.gov:210/lcdb`

常用参数：

| 参数 | 含义 |
|------|------|
| `version` | 协议版本，常用 `1.1` |
| `operation` | `searchRetrieve` 表示检索并取回记录 |
| `query` | **CQL** 检索式（见下文示例） |
| `maximumRecords` | 本页最多返回几条记录 |
| `startRecord` | 从第几条开始（**从 1 起算**） |

响应 XML 根元素为 `searchRetrieveResponse`，其中 `numberOfRecords` 为**总命中数**；每条书目在 `records/record/recordData/record` 下，为 **MARC21 slim** 结构。

**CQL 提示**：可用 `AND` / `OR` 组合；字段检索常用 Dublin Core 索引前缀，例如 `dc.title`、`dc.subject`（是否可用以服务器实际响应为准）。


## 3. MARC 字段速览（本实验常见）

下面读的是 **美国国会图书馆联机目录** 经 SRU 返回的 **MARC 21 XML** 里常见字段。  
MARC 用 **定长字段（controlfield）** 与 **可变长字段（datafield + subfield）** 描述书目。下表侧重「内容 / 出版 / 主题 / 渠道」分析（**非穷尽**；同一条记录不一定全部出现）。

**更完整的官方说明（英文）**：美国国会图书馆维护的 **MARC 21 书目格式（Bibliographic）** 全套字段定义见官网主页 [MARC 21 Format for Bibliographic Data](https://www.loc.gov/marc/bibliographic/)；目录里每个字段常提供 **Full（完整）| Concise（简明）** 两种说明，亦可从导言的简明版入口进入：[Introduction (Concise)](https://www.loc.gov/marc/bibliographic/concise/bdintro.html)。**书目以外**的格式（如规范 Authority、馆藏 Holdings）另有独立文档，可在 [MARC Standards](https://www.loc.gov/marc/) 总入口查找。

| 标签 | 常见含义 |
|------|----------|
| `001` | 记录控制号（LC） |
| `008` | 定长编码数据（出版年、国别、语种等需按 MARC 规则解读） |
| `010` | 美国国会图书馆控制号（LCCN） |
| `020` | ISBN |
| `040` | 编目来源（机构代码） |
| `043` | 地理区划代码 |
| `050` | 美国国会图书馆分类（索书号类号 + 著者号等） |
| `100` / `110` | 主要责任者（个人名 / 团体名） |
| `245` | 题名与责任说明（`$a` 正题名，`$c` 责任者等） |
| `250` | 版本说明 |
| `260` / `264` | 出版发行项（`$a` 出版地，`$b` 出版者，`$c` 出版年） |
| `300` | 载体形态（页码、插图等） |
| `336` / `337` / `338` | 内容类型、媒介类型、载体类型（RDA） |
| `440` / `830` | 丛书 |
| `650` | 主题附加款目—**论题**（`$a` 主题词；地理限定常用 `$z`） |
| `651` | 主题附加款目—**地理名称**（`$a` 地名为主题主入口） |
| `655` | 体裁/文献类型 |
| `856` | 电子资源定位与链接 |

此外 LC 记录中常出现 **`9xx`** 等本地字段（编目流程、馆藏加工），做主题或出版分析时通常可忽略或单独过滤。


### 进入代码区之前

- 确认你已读过上一节的 MARC 表格（不必背下来，运行后面结果时可以翻回来对照）。  
- 下面这一格会**定义函数**（相当于备好工具），**不会立刻联网**；运行成功后通常没有输出或只有空白——这是正常的。  
- 请把代码里的 `USER_AGENT` 联系邮箱改成你自己的学校邮箱（便于馆方识别善意使用）。  
- **建议**：在终端先执行 `cd …/上机实验/01周`，再运行 `jupyter notebook`（或在该文件夹用 VS Code 打开笔记本），这样相对路径 **`data/lc_sru_export/`** 会生成在本周实验目录下。


In [9]:
import time
import csv
import textwrap
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

# ---------- 访问礼貌：对方服务器日志里能看到你是谁 ----------
USER_AGENT = "DigitalLibraryCourse/1.0 (edu-lab; contact: hzhou@ruc.edu.cn)"

# ---------- 美国国会图书馆「联机目录」的 SRU 网关（数据库代号 LCDB）----------
SRU_BASE = "http://lx2.loc.gov:210/lcdb"

# ---------- XML 命名空间：外层「信封」是 SRU(zs)，里面是书目 MARC21 ----------
NS_ZS = {"zs": "http://www.loc.gov/zing/srw/"}
NS_MARC = {"m": "http://www.loc.gov/MARC21/slim"}
MARC_URI = "http://www.loc.gov/MARC21/slim"
DIAG_URI = "http://www.loc.gov/zing/srw/diagnostic/"


def print_section(title: str, subtitle=None, width: int = 76) -> None:
    """控制台大标题：双线框 + 居中主标题；subtitle 可再占一行（例如副标题）。"""
    bar = "=" * width
    print("\n" + bar)
    print(title.center(width))
    if subtitle:
        print(subtitle.center(width))
    print(bar)


def print_subsection(title: str, width: int = 76) -> None:
    """小标题：单线 + 左对齐 ▸，层次比 print_section 弱一档。"""
    print("\n" + "─" * width)
    print(f"  ▸ {title}")
    print("─" * width)


def sru_search(query: str, maximum_records: int = 1, start_record: int = 1, timeout: int = 120) -> bytes:
    """向美国国会图书馆 SRU 网关发「检索并取回记录」请求。

    通俗理解：
    - 就像在浏览器地址栏里访问一长串带参数的网址；这里改成 Python 自动拼 URL、自动下载。
    - `query`：怎么搜——书名里有没有某词、多个词要不要同时出现等（写法叫 CQL）。
    - `maximum_records`：这一趟「最多拿回几条」书目（要做分页就多次调用，每次改起始位置）。
    - `start_record`：从「命中列表里的第几条」开始拿——注意编号从 1 开始，不是编程里常见的 0。

    返回值：整段 XML（二进制 bytes），真正的书目嵌套在深处，后面会用解析函数拆开。
    """
    # 下面这些参数名是 SRU 协议规定好的，不能随便改名，否则服务器不认
    params = {
        "version": "1.1",
        "operation": "searchRetrieve",  # 我们要的操作类型：搜索并返回记录
        "query": query,
        "maximumRecords": str(maximum_records),
        "startRecord": str(start_record),
    }
    # urlencode：把中文、空格、引号等转成 URL 安全的百分号编码
    url = SRU_BASE + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read()


def parse_number_of_records(xml_bytes: bytes) -> int:
    """读出「这次检索一共命中多少条书目」。

    这个数字在 XML 里叫 numberOfRecords——注意它是「总数」，
    未必等于本页实际返回了几条（本页多少条要看 maximumRecords 和是否最后一页）。
    """
    root = ET.fromstring(xml_bytes)
    el = root.find("zs:numberOfRecords", NS_ZS)
    if el is None or el.text is None:
        return 0
    return int(el.text.strip())


def parse_record_count_in_page(xml_bytes: bytes) -> int:
    """数一下：这一份 XML 响应里，SRU 包装层上一共塞了几条「命中」。

    每条命中外面会包一层 <zs:record>；里面才是 MARC。
    翻页时要根据「本页到底回来了几条」去算下一页的起点。
    """
    root = ET.fromstring(xml_bytes)
    return len(root.findall(".//zs:record", NS_ZS))


def parse_next_record_position(xml_bytes: bytes):
    """读取 SRU 响应里的「下一页起点」。最后一页通常没有该元素，返回 None。"""
    root = ET.fromstring(xml_bytes)
    el = root.find("zs:nextRecordPosition", NS_ZS)
    if el is None or el.text is None or not str(el.text).strip():
        return None
    return int(el.text.strip())


def has_diagnostics(xml_bytes: bytes) -> bool:
    """服务器是否在 XML 里塞了「诊断 / 报错」块？

    常见情况：检索式写错、某个运算符不支持、临时故障等。
    有 diagnostics 时通常不应把响应当作正常书目数据继续用。
    """
    root = ET.fromstring(xml_bytes)
    diag = root.find("zs:diagnostics", NS_ZS)
    return diag is not None and len(diag) > 0


def diagnostic_message(xml_bytes: bytes) -> str:
    """把诊断块里的英文说明摘出来拼成一行，方便打印或写进作业报告。"""
    root = ET.fromstring(xml_bytes)
    parts = []
    for d in root.findall(".//{%s}diagnostic" % DIAG_URI):
        m = d.find("{%s}message" % DIAG_URI)
        if m is not None and m.text:
            parts.append(m.text.strip())
    return " | ".join(parts) if parts else ""


def subfield_text(record_el, tag: str, code: str) -> str:
    """在一条 MARC 书目里，按「字段号 + 子字段字母」取文本。

    可以把 MARC 想成很多张卡片格子：
    - `tag`：哪一类格子（三位数，如 245 表示题名项）
    - `code`：这一类下面的细分子格（一个字母，如 a 往往是正题名）

    若同一类格子出现多次，这里只取**第一次**出现的值。
    """
    for df in record_el.findall("m:datafield", NS_MARC):
        if df.get("tag") == tag:
            for sf in df.findall("m:subfield", NS_MARC):
                if sf.get("code") == code and (sf.text or "").strip():
                    return sf.text.strip()
    return ""


def controlfield_text(record_el, tag: str) -> str:
    """读「控制字段」——不分 $a $b，整段就一个字符串。

    常见用途：001（国会图书馆记录号）、008（一串编码位，里面藏出版年、语种等，要查 MARC 手册才能完全读懂）。
    """
    for cf in record_el.findall("m:controlfield", NS_MARC):
        if cf.get("tag") == tag and (cf.text or "").strip():
            return cf.text.strip()
    return ""


def brief_row(record_el) -> dict:
    """把厚厚一条 MARC「压扁」成表格里的一行，方便导出 CSV。

    这只是**抽样字段**，做严肃研究仍应以原始 XML/MARC 为准。
    260 与 264：都是「出版发行」相关著录，老数据多见 260，新规常见 264，所以两个都尝试。
    """
    id001 = controlfield_text(record_el, "001")
    t245 = subfield_text(record_el, "245", "a")
    t260c = subfield_text(record_el, "260", "c") or subfield_text(record_el, "264", "c")
    pub_b = subfield_text(record_el, "260", "b") or subfield_text(record_el, "264", "b")
    pub_a = subfield_text(record_el, "260", "a") or subfield_text(record_el, "264", "a")
    sa = subfield_text(record_el, "050", "a")
    sb = subfield_text(record_el, "050", "b")
    return {
        "001": id001,
        "050": (sa + " " + sb).strip(),
        "245$a": t245,
        "260/264 $a": pub_a,
        "260/264 $b": pub_b,
        "260/264 $c": t260c,
    }


def iter_marc_records(xml_bytes: bytes):
    """遍历当前这一页 XML 里出现的每一条 MARC <record>。

    注意命名空间：同一个 XML 文件里既有 SRU 的标签，也有 MARC 的标签，
    查找时必须带上 MARC 的 URI，避免搞混。
    """
    root = ET.fromstring(xml_bytes)
    yield from root.findall(".//{%s}record" % MARC_URI)

def _marc_local_tag(el) -> str:
    """XML 元素的本地标签名（去掉命名空间花括号）。"""
    t = el.tag
    return t.rsplit("}", 1)[-1] if t.startswith("{") else t


def append_marc_flat_columns(record_el, source_xml: str, cols: dict) -> dict:
    """单次遍历一条 MARC <record>：把各字段追加到 cols 的「列式列表」中（比 list[dict] 再 DataFrame 更省内存、通常更快）。

    cols 的键：bib_id, source_xml, field_order, kind, tag, ind1, ind2, subfield_code, value（均为 list）。
    返回值：与 brief_row 相同结构的 6 列摘要 dict（与原来逻辑一致：260/264 用 OR 合并）。
    """
    bib_id = controlfield_text(record_el, "001")
    order = 0
    used_first_050 = False
    s050a = s050b = None
    t245a = None
    t260a = t260b = t260c = None
    t264a = t264b = t264c = None

    def push(kind: str, tag: str, ind1: str, ind2: str, sfcode: str, val: str) -> None:
        nonlocal order
        cols["bib_id"].append(bib_id)
        cols["source_xml"].append(source_xml)
        cols["field_order"].append(order)
        cols["kind"].append(kind)
        cols["tag"].append(tag)
        cols["ind1"].append(ind1)
        cols["ind2"].append(ind2)
        cols["subfield_code"].append(sfcode)
        cols["value"].append(val)
        order += 1

    for child in record_el:
        ln = _marc_local_tag(child)
        if ln == "leader":
            v = (child.text or "").strip()
            if v:
                push("leader", "LDR", "", "", "", v)
        elif ln == "controlfield":
            tg = child.get("tag", "")
            v = (child.text or "").strip()
            push("control", tg, "", "", "", v)
        elif ln == "datafield":
            tg = child.get("tag", "")
            i1 = (child.get("ind1") or " ")[:1]
            i2 = (child.get("ind2") or " ")[:1]
            first_050 = tg == "050" and not used_first_050
            for sf in child:
                if _marc_local_tag(sf) != "subfield":
                    continue
                code = sf.get("code", "")
                v = (sf.text or "").strip()
                push("data", tg, i1, i2, code, v)
                if first_050:
                    if code == "a" and s050a is None:
                        s050a = v
                    if code == "b" and s050b is None:
                        s050b = v
                if tg == "245" and code == "a" and t245a is None:
                    t245a = v
                if tg == "260":
                    if code == "a" and t260a is None:
                        t260a = v
                    if code == "b" and t260b is None:
                        t260b = v
                    if code == "c" and t260c is None:
                        t260c = v
                if tg == "264":
                    if code == "a" and t264a is None:
                        t264a = v
                    if code == "b" and t264b is None:
                        t264b = v
                    if code == "c" and t264c is None:
                        t264c = v
            if first_050:
                used_first_050 = True

    return {
        "001": bib_id,
        "050": ((s050a or "") + " " + (s050b or "")).strip(),
        "245$a": t245a or "",
        "260/264 $a": (t260a or t264a or ""),
        "260/264 $b": (t260b or t264b or ""),
        "260/264 $c": (t260c or t264c or ""),
    }


## 4. 比较不同检索策略下的命中量（美国国会图书馆 LCDB）

**前置条件**：请先运行上面「导入库与工具函数」那一格代码（定义 `sru_search` 等），否则本格会报错。


下面列出几类典型 CQL 写法（**关键词**、**题名**、**主题**、**组合**等）。每次请求只取 **1 条**书目以减小负载，我们真正关心的是响应里的 **`numberOfRecords`（总命中数）**。

运行下一单元格后，输出按层次组织：**顶部横幅（数据来源说明）→ ① 逐条详情 → ② 一览表 → ③ 小结**，便于截图或课堂展示。

若某条查询返回 **diagnostics**，说明该写法不被服务器接受或索引不可用，可在作业中记录英文提示并换用其他写法。


In [10]:
# (可读标签, CQL 检索式) 
QUERY_CASES = [
    ("关键词（多词同时出现）", "china AND silk AND road"),
    ("题名 dc.title", 'dc.title = "silk road"'),
    ("主题 dc.subject", 'dc.subject = "silk road"'),
    ("题名 + 关键词组合", 'dc.title = "Silk Road" AND china'),
]

PAUSE_SEC = 1.5  # 两次请求之间的间隔，减轻美国国会图书馆服务器压力

rows = []
for label, q in QUERY_CASES:
    try:
        xml = sru_search(q, maximum_records=1, start_record=1)
        total = parse_number_of_records(xml)
        err = ""
        if has_diagnostics(xml):
            err = diagnostic_message(xml) or "diagnostics present"
        rows.append({"策略": label, "CQL": q, "numberOfRecords": total, "备注": err})
    except Exception as e:
        rows.append({"策略": label, "CQL": q, "numberOfRecords": None, "备注": repr(e)})
    time.sleep(PAUSE_SEC)

W = 76
thin = "-" * W

print_section(
    "美国国会图书馆（Library of Congress）",
    "联机目录 LCDB · SRU 检索对比实验",
    width=W,
)
print(
    "\n  【实验目的】比较不同 CQL 写法下的 numberOfRecords（总命中条数）。\n"
    "  【数据出处】http://lx2.loc.gov:210/lcdb（国会图书馆联机目录 SRU）。\n"
    f"  【样本数量】共 {len(rows)} 种检索式；每次只下载 1 条记录以减小负载。\n"
)

print_subsection("① 逐条结果（策略 → 检索式 → 命中）", width=W)
for i, r in enumerate(rows, start=1):
    print(thin)
    print(f"  ({i}/{len(rows)})  {r['策略']}")
    print("      CQL:")
    for line in textwrap.wrap(r["CQL"], width=max(24, W - 14)):
        print(f"          {line}")
    hit = r["numberOfRecords"]
    hit_s = str(hit) if hit is not None else "（请求异常，见备注）"
    print(f"      numberOfRecords ＝  {hit_s}")
    if r["备注"]:
        print(f"      备注 ＝  {r['备注']}")
print(thin)

print_subsection("② 一览表（扫一眼对比规模）", width=W)
print(f"  {'序号':^6}{'命中条数':^12}  策略")
print(thin)
for i, r in enumerate(rows, start=1):
    hit = r["numberOfRecords"]
    hs = str(hit) if hit is not None else "   —"
    flag = "  ⚠" if r["备注"] else ""
    print(f"  {i:^6}{hs:>12}  {r['策略']}{flag}")
print(thin)

print_subsection("③ 小结", width=W)
ok = sum(1 for r in rows if r["numberOfRecords"] is not None and not r["备注"])
print(f"  · 成功返回命中数的检索式：{ok} / {len(rows)}")
print(f"  · 带 diagnostics 或异常的条目：{len(rows) - ok}")
print()

# 若安装了 pandas，可改为表格渲染：
# import pandas as pd
# display(pd.DataFrame(rows))



                        美国国会图书馆（Library of Congress）                        
                           联机目录 LCDB · SRU 检索对比实验                           

  【实验目的】比较不同 CQL 写法下的 numberOfRecords（总命中条数）。
  【数据出处】http://lx2.loc.gov:210/lcdb（国会图书馆联机目录 SRU）。
  【样本数量】共 4 种检索式；每次只下载 1 条记录以减小负载。


────────────────────────────────────────────────────────────────────────────
  ▸ ① 逐条结果（策略 → 检索式 → 命中）
────────────────────────────────────────────────────────────────────────────
----------------------------------------------------------------------------
  (1/4)  关键词（多词同时出现）
      CQL:
          china AND silk AND road
      numberOfRecords ＝  185
----------------------------------------------------------------------------
  (2/4)  题名 dc.title
      CQL:
          dc.title = "silk road"
      numberOfRecords ＝  879
----------------------------------------------------------------------------
  (3/4)  主题 dc.subject
      CQL:
          dc.subject = "silk road"
      numberOfRecords ＝  1148
----------

## 5. 分页获取数据并导出全表（每页 50 条）

**检索策略**：本节的抓取统一使用 **主题索引 `dc.subject`**（示例：`dc.subject = "silk road"`），即在国会图书馆联机目录里按**主题标引**命中的书目集合拉取全量。**不加出版年筛选**。若你想换主题，只需修改下一单元格里的 `MAIN_QUERY` 引号中的措辞。

1. 分页请求：**`maximumRecords=50`** + 优先按响应里的 **`nextRecordPosition`** 推进 **`startRecord`**（若该元素缺失，则用「本页条数」累加），直到收齐 `numberOfRecords`、最后一页不足 50 条、或遇到 SRU 诊断需停止。
2. **若出现诊断 *First record position out of range***：有时表示「请求的起点已超出当前可返回的区段」（可能与网关策略、负载或检索快照有关）。下一单元格遇到该诊断时会**打印说明并停止翻页**，仍导出此前已下载的 XML 与表格；可**稍后再跑**或**改窄检索式**分批抓取。
3. **原始分页 XML** 写入 **`data/lc_sru_export/xml_pages/`**（请先 **`cd` 到 `上机实验/01周`** 再启动 Jupyter）。
4. **导出三类结果**（均需 `pandas` + `pyarrow`）：
   - **`书目摘要.parquet` / `书目摘要.csv`**：仅 6 列速览（001、050、题名、出版项），方便 Excel。
   - **`marc_flat.parquet`**：**长表**，把每条 MARC 的 leader、全部 controlfield、全部 datafield 子字段**逐行展开**（列：`bib_id`, `source_xml`, `field_order`, `kind`, `tag`, `ind1`, `ind2`, `subfield_code`, `value`）。**一本书多行**；含 **650、651** 等全部子字段，可与 XML 一一对应还原。
5. **CSV 说明**：摘要表使用 `utf-8-sig` + `QUOTE_NONNUMERIC`。**完整长表默认只写 Parquet**（行数多）；需要时在代码中取消注释 `marc_flat.csv`。

**性能说明**：长表用 **列式列表** 累积并 **每条书目只遍历 XML 一次**（不再先造大量 `dict` 再 `extend`）。若仍慢，瓶颈多在 **网络下载** 与 **写 Parquet**；可把下一单元格里的 **`EXPORT_MARC_FLAT = False`** 先关掉长表，仅保留 6 列摘要。

**依赖**：`pip install pandas pyarrow`


In [11]:
# 按「主题」检索（dc.subject），不做出版年限制；可改写引号内的主题词
MAIN_QUERY = 'dc.subject = "silk road"'
PAGE_SIZE = 50

# 全部产出放在笔记本所在目录下的 **data** 子文件夹（与本课件并列）
DATA_ROOT = Path("data") / "lc_sru_export"
XML_DIR = DATA_ROOT / "xml_pages"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
XML_DIR.mkdir(parents=True, exist_ok=True)

W = 76
thin = "-" * W

print_section(
    "美国国会图书馆（Library of Congress）",
    "联机目录 LCDB · 分页下载（SRU，每页 50 条）",
    width=W,
)

meta = sru_search(MAIN_QUERY, maximum_records=1, start_record=1)
TOTAL = parse_number_of_records(meta)
if has_diagnostics(meta):
    raise RuntimeError(diagnostic_message(meta))

print_subsection("① 下载参数", width=W)
print(f"  · CQL 检索式:        {MAIN_QUERY}")
print(f"  · 总命中 numberOfRecords:  {TOTAL}")
print(f"  · 每页条数:          {PAGE_SIZE}")
print(f"  · 数据根目录:        {DATA_ROOT.resolve()}")

print_subsection("② 分页进度（XML 按页保存）", width=W)

start = 1
page_idx = 0
all_brief = []
FLAT_KEYS = ("bib_id", "source_xml", "field_order", "kind", "tag", "ind1", "ind2", "subfield_code", "value")
flat_cols = {k: [] for k in FLAT_KEYS}  # 列式累积，避免海量 dict

# 若只要摘要、不要长表，可改为 False（会快一截）
EXPORT_MARC_FLAT = True

while start <= TOTAL:
    page_idx += 1
    print(thin)
    print(f"  第 {page_idx} 页  │  startRecord = {start}")
    blob = sru_search(MAIN_QUERY, maximum_records=PAGE_SIZE, start_record=start)
    if has_diagnostics(blob):
        msg = diagnostic_message(blob)
        if "first record position out of range" in msg.lower():
            print(f"  → SRU 诊断，停止翻页：{msg}")
            print("     （已保留此前各页；可稍后重试，或改窄/拆分检索式。）")
            page_idx -= 1
            break
        raise RuntimeError(msg)
    n = parse_record_count_in_page(blob)
    if n == 0:
        print("  → 本页 0 条，停止。")
        break
    page_path = XML_DIR / f"page_{page_idx:03d}_start{start}.xml"
    page_path.write_bytes(blob)
    print(f"  → 已保存:  {page_path.name}  （本页 {n} 条）")
    for rec in iter_marc_records(blob):
        if EXPORT_MARC_FLAT:
            all_brief.append(append_marc_flat_columns(rec, page_path.name, flat_cols))
        else:
            all_brief.append(brief_row(rec))

    nrp = parse_next_record_position(blob)
    next_start = nrp if nrp is not None else start + n

    if next_start > TOTAL:
        if n < PAGE_SIZE:
            print(f"  → 最后一页（不足 {PAGE_SIZE} 条）。")
        break

    start = next_start
    if n < PAGE_SIZE:
        print(f"  → 最后一页（不足 {PAGE_SIZE} 条）。")
        break
    time.sleep(PAUSE_SEC)

print_subsection("③ 汇总", width=W)
print(f"  · 保存页数:     {page_idx}")
rec_n = len(all_brief)
if rec_n < TOTAL:
    print(f"  · 实际取回书目: {rec_n}（总命中 {TOTAL}；若未收齐，见上文是否因 SRU 诊断提前停止）")
else:
    print(f"  · 解析书目条数: {rec_n}（与总命中 {TOTAL} 一致）")

print_subsection("④ 导出文件（全表）", width=W)

if not all_brief and (not EXPORT_MARC_FLAT or len(flat_cols["bib_id"]) == 0):
    print("  · 未解析到任何书目行，跳过 CSV / Parquet 导出。")
    print()
else:
    try:
        import pandas as pd
    except ImportError as e:
        raise ImportError(
            "导出 Parquet/CSV 需要安装依赖：pip install pandas pyarrow\n"
            "安装后请重新运行本单元格。"
        ) from e

    if all_brief:
        df = pd.DataFrame(all_brief)
        csv_path = DATA_ROOT / "书目摘要.csv"
        parquet_path = DATA_ROOT / "书目摘要.parquet"
        df.to_csv(
            csv_path,
            index=False,
            encoding="utf-8-sig",
            quoting=csv.QUOTE_NONNUMERIC,
        )
        df.to_parquet(parquet_path, index=False, engine="pyarrow")
        print(f"  · Parquet（摘要）: {parquet_path.resolve()}")
        print(f"  · CSV（摘要）:     {csv_path.resolve()}")

    if EXPORT_MARC_FLAT and flat_cols["bib_id"]:
        df_flat = pd.DataFrame(flat_cols)
        flat_parquet = DATA_ROOT / "marc_flat.parquet"
        df_flat.to_parquet(flat_parquet, index=False, engine="pyarrow")
        print(f"  · Parquet（MARC 全字段长表）: {flat_parquet.resolve()}")
        print(f"      行数 {len(df_flat):,}（列式累积 + 每条 record 只遍历 XML 一次）")
        # df_flat.to_csv(DATA_ROOT / "marc_flat.csv", index=False, encoding="utf-8-sig", quoting=csv.QUOTE_NONNUMERIC)

print(f"  · 分页 XML:     {XML_DIR.resolve()} / page_*.xml")
print()



                        美国国会图书馆（Library of Congress）                        
                       联机目录 LCDB · 分页下载（SRU，每页 50 条）                        

────────────────────────────────────────────────────────────────────────────
  ▸ ① 下载参数
────────────────────────────────────────────────────────────────────────────
  · CQL 检索式:        dc.subject = "silk road"
  · 总命中 numberOfRecords:  1148
  · 每页条数:          50
  · 数据根目录:        /Users/hzhou/Library/CloudStorage/OneDrive-s32/教学/2026-数字图书馆前沿/上机实验/01周/data/lc_sru_export

────────────────────────────────────────────────────────────────────────────
  ▸ ② 分页进度（XML 按页保存）
────────────────────────────────────────────────────────────────────────────
----------------------------------------------------------------------------
  第 1 页  │  startRecord = 1
  → 已保存:  page_001_start1.xml  （本页 50 条）
----------------------------------------------------------------------------
  第 2 页  │  startRecord = 51
  → 已保存:  page_002_start51.xml  （本页 50 条）
--

In [13]:
import json
from pathlib import Path

try:
    import pandas as pd
except ImportError as e:
    raise ImportError(
        "本节需要 pandas + pyarrow：pip install pandas pyarrow\n安装后请重新运行本格。"
    ) from e

DATA_ROOT = Path("data") / "lc_sru_export"
FLAT_PATH = DATA_ROOT / "marc_flat.parquet"
OUT_ONE_ROW = DATA_ROOT / "marc_bibli_one_row.parquet"


def flat_group_to_entries(g: pd.DataFrame) -> list[dict]:
    g = g.sort_values("field_order")
    out: list[dict] = []
    for _, r in g.iterrows():
        out.append(
            {
                "field_order": int(r["field_order"]),
                "kind": str(r["kind"]),
                "tag": "" if pd.isna(r["tag"]) else str(r["tag"]),
                "ind1": "" if pd.isna(r["ind1"]) else str(r["ind1"]),
                "ind2": "" if pd.isna(r["ind2"]) else str(r["ind2"]),
                "subfield_code": "" if pd.isna(r["subfield_code"]) else str(r["subfield_code"]),
                "value": "" if pd.isna(r["value"]) else str(r["value"]),
            }
        )
    return out


def entries_to_marc_text(entries: list[dict]) -> str:
    lines: list[str] = []
    i, n = 0, len(entries)
    while i < n:
        e = entries[i]
        k = e["kind"]
        if k == "leader":
            lines.append(f"LDR {e['value']}")
            i += 1
        elif k == "control":
            lines.append(f"{e['tag']} {e['value']}")
            i += 1
        elif k == "data":
            tag = e["tag"]
            i1 = (e["ind1"] or " ")[:1]
            i2 = (e["ind2"] or " ")[:1]
            parts: list[str] = []
            j = i
            while j < n:
                ej = entries[j]
                if ej["kind"] != "data":
                    break
                if ej["tag"] != tag or ej["ind1"] != e["ind1"] or ej["ind2"] != e["ind2"]:
                    break
                code = ej["subfield_code"]
                val = (ej["value"] or "").replace("\n", " ")
                parts.append(f"${code}{val}")
                j += 1
            lines.append(f"={tag} {i1}{i2} " + " ".join(parts))
            i = j
        else:
            lines.append(repr(e))
            i += 1
    return "\n".join(lines)


def build_marc_one_row_table(df_flat: pd.DataFrame) -> pd.DataFrame:
    rows_out: list[dict] = []
    for bib_id, g in df_flat.groupby("bib_id", sort=True):
        src = "|".join(g["source_xml"].drop_duplicates().astype(str))
        entries = flat_group_to_entries(g)
        rows_out.append(
            {
                "bib_id": bib_id,
                "source_xml": src,
                "n_flat_rows": len(entries),
                "marc_json": json.dumps(entries, ensure_ascii=False),
                "marc_text": entries_to_marc_text(entries),
            }
        )
    return pd.DataFrame(rows_out)


def print_full_marc_from_flat(df_flat: pd.DataFrame, bib_id: str) -> None:
    g = df_flat[df_flat["bib_id"].astype(str) == str(bib_id).strip()]
    if g.empty:
        print("未找到 bib_id =", bib_id)
        return
    text = entries_to_marc_text(flat_group_to_entries(g))
    print(f"=== bib_id (001) = {bib_id}  |  长表行数 = {len(g)} ===\n")
    print(text)


def print_full_marc_from_one_row(row: pd.Series) -> None:
    bid = row["bib_id"]
    print(f"=== bib_id (001) = {bid}  |  n_flat_rows = {row['n_flat_rows']} ===\n")
    print(row["marc_text"])


if not FLAT_PATH.exists():
    print("未找到:", FLAT_PATH.resolve())
    print("请确认 §5 已运行且 EXPORT_MARC_FLAT = True。")
else:
    df_flat_disk = pd.read_parquet(FLAT_PATH)
    df_bib = build_marc_one_row_table(df_flat_disk)
    df_bib.to_parquet(OUT_ONE_ROW, index=False)
    print("已写出:", OUT_ONE_ROW.resolve())
    print("书目数 × 列数:", df_bib.shape)

    SHOW_BIB_ID = str(df_bib["bib_id"].iloc[0])
    print("\n--- 示例：完整 MARC 文本（bib_id =", SHOW_BIB_ID, "）---\n")
    print_full_marc_from_flat(df_flat_disk, SHOW_BIB_ID)
    

已写出: /Users/hzhou/Library/CloudStorage/OneDrive-s32/教学/2026-数字图书馆前沿/上机实验/01周/data/lc_sru_export/marc_bibli_one_row.parquet
书目数 × 列数: (1141, 5)

--- 示例：完整 MARC 文本（bib_id = 10002262 ）---

=== bib_id (001) = 10002262  |  长表行数 = 42 ===

LDR 00763cam a2200265u  4500
001 10002262
005 20250606153704.0
008 860108s1979    xx            000 0 jpn
=035    $a10002262
=906    $a0 $bcbc $cpremunv $du $encip $f19 $gy-gencatlg
=955    $aLCAP batch update 2025-06-01-04:00: LCAPM-644
=010    $a80801453
=035    $9(DLC)   80801453
=040    $aDLC $cCarP $dDLC
=050 00 $aDS786 $b.E54
=100 1  $aEnoki, Kazuo, $d1913-1989.
=245 10 $aShiruku Rōdo no rekishi kara.
=260    $c1979.
=300    $a230 p. ; $c20 cm.
=336    $atext $btxt $2rdacontent
=337    $aunmediated $bn $2rdamedia
=338    $avolume $bnc $2rdacarrier
=500    $aRomanized.
=651  0 $aSilk Road.
=991    $bc-GenColl $hDS786 $i.E54 $tCopy 1 $wPREM


## 6. 快速查看前几行摘要





In [18]:
import random

has650 = df_flat_disk[
    (df_flat_disk["kind"] == "data") & (df_flat_disk["tag"] == "650")
]["bib_id"].astype(str).unique().tolist()

if not has650:
    print("当前库里没有 650 字段（不应该）；请检查 marc_flat")
else:
    bid = random.choice(has650)
    print(f"随机抽到（含 650）bib_id = {bid}\n")
    print_full_marc_from_flat(df_flat_disk, bid)

随机抽到（含 650）bib_id = 13013071

=== bib_id (001) = 13013071  |  长表行数 = 88 ===

LDR 02156cam a22004574a 4500
001 13013071
005 20250606195240.2
008 021125s2003    ohua     bc   000 0 eng
=035    $a13013071
=906    $a7 $bcbc $corignew $d1 $eecip $f20 $gy-gencatlg
=925 0  $aacquire $b1 shelf copy $xpolicy default
=925 1  $aacquire $b2 shelf copies $xpolicy default
=955    $ajh26 2002-11-25 $aaa08 2002-11-27 $aps11 2003-04-03 1 copy rec'd., to CIP ver.; $ajh00 2003-04-07; $ajh85 2004-07-23 to BCCD (copy 2) $cjh26 2002-11-25 $cjh32 2003-06-10 to SL $djh34 2002-11-25 to SL $ejh85 2002-11-26 to Dewey $gjh85 2003-06-11 to BCCD $aLCAP batch update 2025-06-01-04:00: LCAPM-644, LCAPM-893, LCAPM-150
=010    $a2002154401
=020    $a0937809241 $a9780937809242
=040    $aDLC $cDLC $dDLC
=042    $apcc
=043    $aa-cc---
=050 00 $aN7343.24 $b.G57 2003
=082 00 $a709/.51/07476819 $221
=245 04 $aThe glory of the Silk Road : $bart from ancient China / $cLi Jian, general editor ; with essays by Valerie Hansen ...

In [25]:
import json
import re
from pathlib import Path

import pandas as pd

DATA_ROOT = Path("data") / "lc_sru_export"
df_one = pd.read_parquet(DATA_ROOT / "marc_bibli_one_row.parquet")

def join_unique(values, sep=" | "):
    out, seen = [], set()
    for v in values:
        if v is None or (isinstance(v, float) and pd.isna(v)):
            continue
        s = str(v).strip()
        if not s or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return sep.join(out) if out else None

def first_value(values):
    for v in values:
        if v is None or (isinstance(v, float) and pd.isna(v)):
            continue
        s = str(v).strip()
        if s:
            return s
    return None

year_re = re.compile(r"((?:1|2)\d{3})")

records = []
for _, r in df_one.iterrows():
    bib_id = str(r["bib_id"])
    fields = json.loads(r["marc_json"])  # list[dict]

    # 收集：datafield 用 (tag, sub)；controlfield 用 tag（例如 008）
    data_map = {}   # (tag, sub) -> list[str]
    ctrl_map = {}   # tag -> list[str]

    for f in fields:
        kind = f.get("kind")
        tag = f.get("tag")
        if not tag:
            continue

        if kind == "data":
            sub = f.get("subfield_code")
            val = f.get("value")
            if not sub:
                continue
            data_map.setdefault((tag, sub), []).append(val)
        elif kind == "control":
            ctrl_map.setdefault(tag, []).append(f.get("value"))

    def allv(tag, sub):
        return data_map.get((tag, sub), [])

    def first(tag, sub):
        return first_value(allv(tag, sub))

    def join(tag, sub, sep=" | "):
        return join_unique(allv(tag, sub), sep=sep)

    # 008：出版国家/语言（位置字段）
    v008 = first_value(ctrl_map.get("008", []))
    country_008 = v008[15:18] if isinstance(v008, str) and len(v008) >= 18 else None
    lang_008 = v008[35:38] if isinstance(v008, str) and len(v008) >= 38 else None

    # 出版字段：优先 264，没有就用 260
    has_264 = any(k[0] == "264" for k in data_map.keys())
    pub_tag = "264" if has_264 else "260"
    pub_place = join(pub_tag, "a")
    publisher = join(pub_tag, "b")
    pub_date_raw = join(pub_tag, "c")

    pub_year = None
    if pub_date_raw:
        m = year_re.search(pub_date_raw)
        pub_year = int(m.group(1)) if m else None

    # 050 组合成一个分类号字符串
    t050a = first("050", "a")
    t050b = first("050", "b")
    lc_class_050 = (" ".join([x for x in [t050a, t050b] if x]).strip() or None)

    records.append({
        "bib_id": bib_id,

        # 出版国家/语言（用于结构分析很关键）
        "country_008": country_008,     # 008/15-17 国家代码（如 "cc " / "xx " 等）
        "lang_008": lang_008,           # 008/35-37 语言代码
        "langs_041$a": join("041", "a"),# 多语种时很有用（可能为空）

        # 标识符（去重/链接外部数据）
        "lccn_010$a": join("010", "a"),
        "isbn_020$a": join("020", "a"),
        "issn_022$a": join("022", "a"),

        # 题名/责任者（内容分析入口）
        "title_245$a": first("245", "a"),
        "title_245$b": first("245", "b"),
        "resp_245$c": first("245", "c"),

        # 作者/附加作者（网络分析）
        "author_100$a": first("100", "a"),
        "contributors_700$a": join("700", "a"),

        # 分类/出版（馆藏结构）
        "lc_class_050": lc_class_050,
        "pub_place": pub_place,
        "publisher": publisher,
        "pub_date_raw": pub_date_raw,
        "pub_year": pub_year,

        # 载体/资源类型（馆藏形态）
        "extent_300$a": join("300", "a"),
        "content_336$a": join("336", "a"),
        "media_337$a": join("337", "a"),
        "carrier_338$a": join("338", "a"),

        # 内容提要/目录（内容分析强信号，可能有些记录为空）
        "abstract_520$a": join("520", "a", sep="\n"),
        "toc_505$a": join("505", "a", sep="\n"),

        # 主题/地理/体裁（你关心的 650 地理就在 650$z）
        "subject_650$a": join("650", "a"),
        "subject_650$x": join("650", "x"),
        "subject_650$y": join("650", "y"),
        "subject_650$z": join("650", "z"),  # 650 里的地理信息关键列
        "geo_651$a": join("651", "a"),
        "genre_655$a": join("655", "a"),

        # 外部链接
        "url_856$u": join("856", "u"),
    })

df_parsed = pd.DataFrame(records)

out_path = DATA_ROOT / "marc_parsed_analysis.parquet"
df_parsed.to_parquet(out_path, index=False)

display(df_parsed), out_path


,bib_id,country_008,lang_008,langs_041$a,lccn_010$a,isbn_020$a,issn_022$a,title_245$a,title_245$b,resp_245$c,...,carrier_338$a,abstract_520$a,toc_505$a,subject_650$a,subject_650$x,subject_650$y,subject_650$z,geo_651$a,genre_655$a,url_856$u
0,10002262,xx,jpn,None,80801453,None,None,Shiruku Rōdo no rekishi kara.,None,None,...,volume,None,None,None,None,None,None,Silk Road.,None,None
1,10008602,ja,jpn,None,81801533,None,None,Sanzō Hōshi no michi :,Shiruku Rōdo kikō /,Chin Sunshin bun ; Chin Ritsujin shashin.,...,volume,None,None,Travel.,None,None,None,"Asia, Central | Silk Road | Asia | Central Asia.",None,None
2,10009061,xx,jpn,None,81806393,None,None,Yūkyū no nagare no naka ni.,None,None,...,volume,None,None,"Painters | Buddhist art, in art.",None,None,Japan,Silk Road,Biographies.,None
3,10017793,xx,jpn,None,82803053,None,None,Rōma kara Yamato e.,None,None,...,volume,None,None,Artists' preparatory studies,None,None,Japan.,Asia | Europe | Silk Road,None,None
4,10431714,xx,jpn,None,75810202,None,None,Hekiga no bijo.,None,None,...,volume,None,None,Mural painting and decoration | Women in art. ...,None,None,Asia.,Silk Road.,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1136,in00024350060,cc,chi,None,2025407944,9787549023349 | 7549023344,None,Si chou zhi lu yu Zhong wai guan xi shi zhu xi...,None,Yang Fuxue zhu.,...,volume,None,None,None,None,None,None,China | Silk Road | Chine | Route de la soie,None,None
1137,in00024350125,cc,chi,None,2025407949,9787549026333 | 7549026335,None,Si chou zhi lu gu qian bi yan jiu /,None,"Yang Fuxue, Yuan Wei zhu bian.",...,volume,None,None,"Coins, Ancient | Monnaies antiques",None,None,Silk Road. | Route de la soie.,Silk Road.,None,None
1138,in00024414901,nyu,eng,None,2026349047,9798217126514,None,Silk roads :,a flavor odyssey with recipes from Baku to Bei...,Anna Ansari.,...,volume,"""This is the story of the world's most famous ...",None,"Cooking, Asian | Trade routes | Cuisine asiati...",None,None,Asia | Asie,Silk Road | Route de la soie,Cookbooks | Travel writing | cookbooks | Livre...,None
1139,in00024518479,ko,kor,None,2025477635,9791191670653,None,Taesŭng Pulgyo nŭn ŏttŏk'e palchŏn haennŭn'ga /,None,Han Chi-yŏn.,...,volume,None,None,Mahayana Buddhism | Buddhism | Bouddhisme Mahā...,History | Histoire,None,India | China | Inde | Chine,Silk Road | Route de la soie,History,None


(None, PosixPath('data/lc_sru_export/marc_parsed_analysis.parquet'))